# Vehicle Tracking, Counting and Congestion Estimation

Detects vehicles with YOLOv8x, tracks them across frames with ByteTrack, and
estimates per-zone counts, individual speeds and an overall congestion level
(graded A-F) from a fixed surveillance camera.

Part of a final-year project on video analytics for traffic/surveillance footage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
SOURCE_VIDEO_PATH = "/content/drive/MyDrive/hospital_video/car1.mp4"

In [ ]:
# Pin ultralytics to a version compatible with the older supervision API used here
!pip install "ultralytics<=8.3.40"

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

In [ ]:
%cd {HOME}
!git clone https://github.com/ifzhang/ByteTrack.git
%cd {HOME}/ByteTrack

!sed -i 's/onnx==1.8.1/onnx==1.9.0/g' requirements.txt

!pip3 install -q -r requirements.txt
!python3 setup.py -q develop
!pip install -q cython_bbox
!pip install -q onemetric
!pip install -q loguru lap thop

from IPython import display
display.clear_output()

import sys
sys.path.append(f"{HOME}/ByteTrack")

import yolox
print("yolox.__version__:", yolox.__version__)

In [ ]:
from yolox.tracker.byte_tracker import BYTETracker, STrack
from onemetric.cv.utils.iou import box_iou_batch
from dataclasses import dataclass


@dataclass(frozen=True)
class BYTETrackerArgs:
    track_thresh: float = 0.25
    track_buffer: int = 30
    match_thresh: float = 0.8
    aspect_ratio_thresh: float = 3.0
    min_box_area: float = 1.0
    mot20: bool = False

In [ ]:
!pip install supervision==0.1.0

from IPython import display
display.clear_output()

import supervision
print("supervision.__version__:", supervision.__version__)

In [ ]:
from supervision.draw.color import ColorPalette
from supervision.geometry.dataclasses import Point
from supervision.video.dataclasses import VideoInfo
from supervision.video.source import get_video_frames_generator
from supervision.video.sink import VideoSink
from supervision.notebook.utils import show_frame_in_notebook
from supervision.tools.detections import Detections, BoxAnnotator
from supervision.tools.line_counter import LineCounter, LineCounterAnnotator

In [ ]:
from typing import List

import numpy as np

def detections2boxes(detections: Detections) -> np.ndarray:
    return np.hstack((
        detections.xyxy,
        detections.confidence[:, np.newaxis]
    ))


def tracks2boxes(tracks: List[STrack]) -> np.ndarray:
    return np.array([
        track.tlbr
        for track
        in tracks
    ], dtype=np.float64)


def match_detections_with_tracks(
    detections: Detections,
    tracks: List[STrack]
) -> Detections:
    if not np.any(detections.xyxy) or len(tracks) == 0:
        return np.empty((0,))

    tracks_boxes = tracks2boxes(tracks=tracks)
    iou = box_iou_batch(tracks_boxes, detections.xyxy)
    track2detection = np.argmax(iou, axis=1)

    tracker_ids = [None] * len(detections)

    for tracker_index, detection_index in enumerate(track2detection):
        if iou[tracker_index, detection_index] != 0:
            tracker_ids[detection_index] = tracks[tracker_index].track_id

    return tracker_ids

In [ ]:
MODEL = "yolov8x.pt"

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.fuse()

In [ ]:
# Map class_id -> class_name, then keep only vehicle classes (car, bus, truck)
CLASS_NAMES_DICT = model.model.names
CLASS_ID = [2, 5, 7]

In [ ]:
generator = get_video_frames_generator(SOURCE_VIDEO_PATH)
box_annotator = BoxAnnotator(color=ColorPalette(), thickness=4, text_thickness=4, text_scale=2)
iterator = iter(generator)
frame = next(iterator)
results = model(frame)
detections = Detections(
    xyxy=results[0].boxes.xyxy.cpu().numpy(),
    confidence=results[0].boxes.conf.cpu().numpy(),
    class_id=results[0].boxes.cls.cpu().numpy().astype(int)
)
labels = [
    f"{CLASS_NAMES_DICT[class_id]} {confidence:0.2f}"
    for _, confidence, class_id, tracker_id
    in detections
]
frame = box_annotator.annotate(frame=frame, detections=detections, labels=labels)

%matplotlib inline
show_frame_in_notebook(frame, (16, 16))

In [ ]:
LINE_START = Point(0, 540)
LINE_END = Point(1920, 540)

TARGET_VIDEO_PATH = f"{HOME}/vehicle-counting-result.mp4"

In [ ]:
VideoInfo.from_video_path(SOURCE_VIDEO_PATH)

In [ ]:
# ByteTrack's bundled utilities expect the older NumPy API
!pip install "numpy<1.24"

In [ ]:
np.__version__

In [ ]:
import numpy as np
import cv2
import time
from collections import defaultdict
from typing import List, Optional
from dataclasses import dataclass
import os
import math
from tqdm.notebook import tqdm
from ultralytics import YOLO
from supervision.draw.color import ColorPalette
from supervision.geometry.dataclasses import Point
from supervision.video.dataclasses import VideoInfo
from supervision.video.source import get_video_frames_generator
from supervision.video.sink import VideoSink
from supervision.tools.detections import Detections, BoxAnnotator
from yolox.tracker.byte_tracker import BYTETracker, STrack
from onemetric.cv.utils.iou import box_iou_batch


class STrack(object):
    def __init__(self, tlwh, score):
        self._tlwh = np.asarray(tlwh, dtype=np.float64)
        self.kalman_filter = None
        self.mean, self.covariance = None, None


class AreaCounter:
    """Counts vehicles inside a polygon zone and grades congestion from their speeds."""

    def __init__(self, area_boundary: List[tuple], pixel_per_meter=10):
        self.area_boundary = np.array(area_boundary, dtype=np.int32)
        self.area_boundary = self.area_boundary.reshape((-1, 1, 2))

        self.kalman_filter = None
        self.mean, self.covariance = None, None

        self.vehicles_inside = {}
        self.vehicles_history = defaultdict(list)
        self.entry_count = 0
        self.exit_count = 0
        self.speeds = {}
        self.pixel_per_meter = pixel_per_meter

    def is_inside(self, x, y):
        point = np.array([x, y], dtype=np.float32)
        return cv2.pointPolygonTest(self.area_boundary, tuple(point), False) >= 0

    def update(self, detections, speed_estimator=None):
        current_time = time.time()
        current_vehicles = set()

        for i in range(len(detections.xyxy)):
            xyxy = detections.xyxy[i]
            tracker_id = detections.tracker_id[i]

            if tracker_id is None:
                continue

            center_x = (xyxy[0] + xyxy[2]) / 2
            center_y = (xyxy[1] + xyxy[3]) / 2

            if self.is_inside(center_x, center_y):
                current_vehicles.add(tracker_id)

                # New vehicle entering the zone
                if tracker_id not in self.vehicles_inside:
                    self.vehicles_inside[tracker_id] = current_time
                    self.entry_count += 1
                    self.vehicles_history[tracker_id].append(("entered", current_time))

                if speed_estimator:
                    speed = speed_estimator.get_speed(tracker_id)
                    if speed is not None:
                        self.speeds[tracker_id] = speed

            elif tracker_id in self.vehicles_inside:
                # Vehicle has left the zone
                del self.vehicles_inside[tracker_id]
                self.exit_count += 1
                self.vehicles_history[tracker_id].append(("exited", current_time))

                if tracker_id in self.speeds:
                    del self.speeds[tracker_id]

    def get_counts(self):
        return {
            "inside": len(self.vehicles_inside),
            "entered": self.entry_count,
            "exited": self.exit_count
        }

    def get_average_speed(self):
        if not self.speeds:
            return 0
        return sum(self.speeds.values()) / len(self.speeds)

    def get_all_vehicle_speeds(self):
        return self.speeds.copy()

    def get_congestion_level(self):
        # Grade A (free flow) through F (jammed), keyed off the average speed
        avg_speed = self.get_average_speed()

        if avg_speed >= 35:
            return "A"
        elif 30 <= avg_speed < 35:
            return "B"
        elif 25 <= avg_speed < 30:
            return "C"
        elif 20 <= avg_speed < 25:
            return "D"
        elif 15 <= avg_speed < 20:
            return "E"
        else:
            return "F"

    def draw_area(self, frame, color=(0, 255, 0), thickness=2):
        cv2.polylines(frame, [self.area_boundary], isClosed=True, color=color, thickness=thickness)

        counts = self.get_counts()
        avg_speed = self.get_average_speed()
        congestion_level = self.get_congestion_level()
        x_min, y_min = self.area_boundary[0][0]

        text = f"Inside: {counts['inside']}  In: {counts['entered']}  Out: {counts['exited']}  Avg Speed: {avg_speed:.1f} km/h"
        cv2.putText(frame, text, (int(x_min), int(y_min) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        congestion_text = f"Congestion Level: {congestion_level}"
        cv2.putText(frame, congestion_text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

        return frame


class SpeedEstimator:
    def __init__(self, fps, pixel_per_meter=10):
        self.positions = {}  # tracker_id -> [(x, y, timestamp), ...]
        self.speeds = {}
        self.fps = fps
        self.pixel_per_meter = pixel_per_meter

    def update(self, detections):
        current_time = time.time()

        for i in range(len(detections.xyxy)):
            xyxy = detections.xyxy[i]
            tracker_id = detections.tracker_id[i]
            if tracker_id is not None:
                center_x = (xyxy[0] + xyxy[2]) / 2
                center_y = (xyxy[1] + xyxy[3]) / 2

                if tracker_id not in self.positions:
                    self.positions[tracker_id] = []

                self.positions[tracker_id].append((center_x, center_y, current_time))

                # Keep only the last few positions per track
                if len(self.positions[tracker_id]) > 10:
                    self.positions[tracker_id] = self.positions[tracker_id][-10:]

                if len(self.positions[tracker_id]) >= 2:
                    self._calculate_speed(tracker_id)

    def _calculate_speed(self, tracker_id):
        positions = self.positions[tracker_id]
        if len(positions) < 2:
            return

        first_pos = positions[0]
        last_pos = positions[-1]

        time_diff = last_pos[2] - first_pos[2]
        if time_diff <= 0:
            time_diff = 1 / self.fps  # guard against division by zero

        distance_pixels = math.sqrt((last_pos[0] - first_pos[0]) ** 2 + (last_pos[1] - first_pos[1]) ** 2)
        distance_meters = distance_pixels / self.pixel_per_meter

        speed_mps = distance_meters / time_diff
        speed_kph = speed_mps * 3.6
        self.speeds[tracker_id] = speed_kph

    def get_speed(self, tracker_id):
        if tracker_id in self.speeds:
            return self.speeds[tracker_id]
        return None


@dataclass(frozen=True)
class BYTETrackerArgs:
    track_thresh: float = 0.25
    track_buffer: int = 30
    match_thresh: float = 0.8
    aspect_ratio_thresh: float = 3.0
    min_box_area: float = 1.0
    mot20: bool = False


def detections2boxes(detections: Detections) -> np.ndarray:
    return np.hstack((
        detections.xyxy,
        detections.confidence[:, np.newaxis]
    ))


def tracks2boxes(tracks: List[STrack]) -> np.ndarray:
    return np.array([
        track.tlbr
        for track
        in tracks
    ], dtype=np.float64)


def match_detections_with_tracks(
        detections: Detections,
        tracks: List[STrack]
) -> Detections:
    if not np.any(detections.xyxy) or len(tracks) == 0:
        return np.empty((0,))

    tracks_boxes = tracks2boxes(tracks=tracks)
    iou = box_iou_batch(tracks_boxes, detections.xyxy)
    track2detection = np.argmax(iou, axis=1)

    tracker_ids = [None] * len(detections)

    for tracker_index, detection_index in enumerate(track2detection):
        if iou[tracker_index, detection_index] != 0:
            tracker_ids[detection_index] = tracks[tracker_index].track_id

    return tracker_ids


def main():
    byte_tracker = BYTETracker(BYTETrackerArgs())
    video_info = VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
    generator = get_video_frames_generator(SOURCE_VIDEO_PATH)

    # Polygon marking the monitored area - adjust to the video resolution
    AREA_BOUNDARY = [(680, 30), (1330, 30), (1980, 950), (210, 950)]

    area_counter = AreaCounter(area_boundary=AREA_BOUNDARY)
    speed_estimator = SpeedEstimator(fps=video_info.fps)
    box_annotator = BoxAnnotator(color=ColorPalette(), thickness=4, text_thickness=4, text_scale=2)

    with VideoSink(TARGET_VIDEO_PATH, video_info) as sink:
        for frame in tqdm(generator, total=video_info.total_frames):
            results = model(frame)
            detections = Detections(
                xyxy=results[0].boxes.xyxy.cpu().numpy(),
                confidence=results[0].boxes.conf.cpu().numpy(),
                class_id=results[0].boxes.cls.cpu().numpy().astype(int)
            )

            # Keep only vehicle classes
            mask = np.array([class_id in CLASS_ID for class_id in detections.class_id], dtype=bool)
            detections.filter(mask=mask, inplace=True)

            tracks = byte_tracker.update(
                output_results=detections2boxes(detections=detections),
                img_info=frame.shape,
                img_size=frame.shape
            )
            tracker_id = match_detections_with_tracks(detections=detections, tracks=tracks)
            detections.tracker_id = np.array(tracker_id)

            # Drop detections that were not matched to a track
            mask = np.array([tracker_id is not None for tracker_id in detections.tracker_id], dtype=bool)
            detections.filter(mask=mask, inplace=True)

            speed_estimator.update(detections)
            area_counter.update(detections, speed_estimator)

            labels = []
            for i, (confidence, class_id, tracker_id) in enumerate(zip(detections.confidence, detections.class_id, detections.tracker_id)):
                speed = speed_estimator.get_speed(tracker_id)
                if speed is not None:
                    labels.append(f"#{tracker_id} {CLASS_NAMES_DICT[class_id]} {confidence:0.2f} {speed:.1f}km/h")
                else:
                    labels.append(f"#{tracker_id} {CLASS_NAMES_DICT[class_id]} {confidence:0.2f}")

            frame = box_annotator.annotate(frame=frame, detections=detections, labels=labels)
            frame = area_counter.draw_area(frame)
            sink.write_frame(frame)


if __name__ == "__main__":
    HOME = os.getcwd()
    SOURCE_VIDEO_PATH = "/content/drive/MyDrive/hospital_video/car1.mp4"
    MODEL = "yolov8x.pt"
    TARGET_VIDEO_PATH = f"{HOME}/vehicle-counting-result.mp4"

    model = YOLO(MODEL)
    model.fuse()
    CLASS_NAMES_DICT = model.model.names
    CLASS_ID = [2, 5, 7]  # car, bus, truck

    main()